# Topic 10 課堂實作：Transfer Learning × 工業瑕疵檢測（MVTec AD）

**CISE2181 Machine Learning · Fall 2026 · ST301 GenAI GPU Classroom**

這份 notebook 把 Topic 10 的內容（CNN、預訓練模型、Transfer Learning、VGG16 / MobileNet / ResNet / EfficientNet）直接接到期中專案 **Track A：Supervised classifier** 的流程上。跑完之後，你會有一套可以直接延伸到期中專案的實驗骨架。

### 今天的學習目標

| # | 你會做的事 | 對應 Topic 10 / 期中 |
|---|---|---|
| 1 | 依期中規定把 MVTec AD 官方 test set 做 50/50 分層切分，並留一份**絕不碰**的 held-out | 期中：Experimental design |
| 2 | 打開預訓練 CNN：看第一層 filter、各層 feature map、ImageNet 特徵的分布 | CNN Visualized |
| 3 | 比較 **從零訓練 (scratch)** vs **凍結特徵 (frozen)** vs **微調 (fine-tune)** | Transfer Learning (C3W2L07) |
| 4 | 換 backbone：ResNet-18 / MobileNetV3 / EfficientNet-B0 / VGG16 | Transfer learning explained |
| 5 | 用 defect mask **量化**哪些 augmentation 會讓標籤失真 | 期中：Augmentation study |
| 6 | AUROC、Recall@5%FPR、混淆矩陣、Grad-CAM 錯誤分析 | 期中：Evaluation / Error analysis |

### 建議課堂時間配置（約 150 分鐘）

| 段落 | 時間 | 備註 |
|---|---|---|
| §0–§3 環境、資料、切分 | 25 min | 資料請**課前**準備好（見 §2） |
| §4–§5 打開預訓練 CNN | 20 min | |
| §6–§7 三種訓練策略 | 35 min | 核心段落 |
| §8 Backbone 比較 | 15 min | 時間不夠可留作業 |
| §9 Augmentation 與標籤失真 | 30 min | 期中分數最多的地方 |
| §10–§11 門檻、錯誤分析 | 20 min | |
| §12 存檔、練習 | 5 min | |

### 執行環境
- **Google Colab**：`執行階段 → 變更執行階段類型 → GPU (T4 即可)`。
- **ST301 本機（RTX PRO 4000 Blackwell, 24 GB）**：用 **uv** 建立環境（不要用 `pip install`）。在 cmd 中：

```bat
D:
cd \cise2181\cise2181-starter
uv sync
uv run jupyter lab
```

`uv sync` 會依 repo 裡的 `pyproject.toml` 與 `uv.lock` 安裝完全相同版本的套件（含支援 Blackwell 的 PyTorch CUDA 12.8 版）。

> Colab 已內建 `torch`、`torchvision`、`scikit-learn`、`pandas`、`matplotlib`，不需要 uv，也不需要額外安裝。

## §0 環境檢查

先確認有沒有 GPU、PyTorch 版本是否支援這張卡。

**ST301 本機注意**：RTX PRO 4000 Blackwell 的 compute capability 是 12.0（sm_120），需要 **PyTorch ≥ 2.7 搭配 CUDA 12.8 以上**的版本。課程 repo 的 `pyproject.toml` 已經指定好；若下面的檢查出現警告，代表你不是用 `uv run jupyter lab` 啟動的。請關掉 Jupyter，回到 cmd 執行：

```bat
uv sync
uv run jupyter lab
```

並確認 JupyterLab 右上角的 kernel 是這個 repo 的 `.venv`。

In [ ]:
import sys, os, platform, time, json, random, math, tarfile, urllib.request, warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch, torchvision
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T

print(f"Python      : {platform.python_version()}  ({'Colab' if IN_COLAB else platform.system()})")
print(f"PyTorch     : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU         : {name}  (compute capability {cap[0]}.{cap[1]}, {mem:.1f} GB)")
    arch_list = torch.cuda.get_arch_list()
    if f"sm_{cap[0]}{cap[1]}" not in arch_list and cap[0] >= 12:
        print("⚠️ 這個 PyTorch 版本沒有編譯 sm_%d%d，Blackwell 可能無法執行。請在 cmd 執行 uv sync 後，用 uv run jupyter lab 重新啟動。" % cap)
    # 小測試：確認 kernel 真的能在這張卡上跑
    _ = (torch.randn(64, 64, device=DEVICE) @ torch.randn(64, 64, device=DEVICE)).sum().item()
    print("GPU 測試     : OK")
else:
    DEVICE = torch.device("cpu")
    print("⚠️ 沒有偵測到 GPU，會用 CPU 執行（可以跑，但會比較慢）。")

In [ ]:
def set_seed(seed: int):
    # 固定所有亂數來源：Python / NumPy / PyTorch(CPU+GPU)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.dpi"] = 100
set_seed(0)

## §1 課堂設定

所有實驗設定都集中在這一格。**期中專案的原則：一次只改一件事，其他保持固定。**

- 課堂示範使用 2 個類別：`tile`（texture）與 `transistor`（object）。期中請改成**你們登記的 3 個類別**（至少 1 texture + 1 object）。
- `QUICK_MODE = True` 時只跑 1 個 seed，方便課堂上快速看到結果；期中報告必須用 3 個 seed（`QUICK_MODE = False`）。
- 若要更快，可把 `transistor` 換成較小的 `bottle`。

In [ ]:
# ===================== 課堂設定 =====================
CATEGORIES  = ["tile", "transistor"]   # 期中：換成你們登記的 3 個類別
IMG_SIZE    = 224                      # 投影片建議 224–256
CACHE_SIZE  = 256                      # 圖片先縮到這個大小放進記憶體，加快訓練
BATCH_SIZE  = 16
EPOCHS      = 30
PATIENCE    = 6                        # early stopping：val loss 連續幾個 epoch 沒進步就停
SPLIT_SEED  = 42                       # 資料切分的 seed：固定不動！
QUICK_MODE  = True                     # 課堂 True（1 seed）；期中報告 False（3 seeds）
SEEDS       = [0] if QUICK_MODE else [0, 1, 2]
TARGET_FPR  = 0.05                     # 「只允許 5% 的良品被誤判」的條件
NUM_WORKERS = 0                        # 資料已快取在記憶體，0 最穩定（Windows/Jupyter 也不會出錯）

OUT_DIR = Path("outputs_topic10"); OUT_DIR.mkdir(exist_ok=True)
SPLIT_DIR = OUT_DIR / "splits";    SPLIT_DIR.mkdir(exist_ok=True)
print("seeds =", SEEDS, "| device =", DEVICE, "| outputs ->", OUT_DIR.resolve())

## §2 準備 MVTec AD 資料

MVTec AD 授權為 **CC BY-NC-SA 4.0**（可用於課程與研究，不可商用；引用 Bergmann et al., CVPR 2019 / IJCV 2021）。下載請從官方頁面 mvtec.com → Research → Datasets 取得。

完整資料約 5 GB，**請不要 40 個人同時在課堂上下載**。建議做法：

| 環境 | 建議做法 | 要設定的變數 |
|---|---|---|
| ST301 本機 | 助教事先解壓到共用路徑，例如 `D:/datasets/mvtec_ad` | `DATA_ROOT` 或環境變數 `MVTEC_ROOT` |
| Colab | 把壓縮檔（完整版或單一類別的 `tile.tar.xz`）放在自己的 Google Drive | `MVTEC_TAR` |
| 任何環境 | 老師提供的直連網址 | `MVTEC_URL` |

解壓後的資料夾結構應該長這樣（`DATA_ROOT` 指向最上層）：

```
DATA_ROOT/
├── tile/
│   ├── train/good/*.png
│   ├── test/good/*.png, test/crack/*.png, ...
│   └── ground_truth/crack/*_mask.png, ...
└── transistor/ ...
```

In [ ]:
# ---- 依你的環境修改其中一項（全部留空 = 自動搜尋常見路徑） ----
DATA_ROOT = None   # 例如 "D:/datasets/mvtec_ad"
MVTEC_TAR = None   # 例如 "/content/drive/MyDrive/datasets/mvtec_anomaly_detection.tar.xz"
MVTEC_URL = ""     # 例如老師提供的直連網址
EXTRACT_TO = Path("/content/mvtec_ad") if IN_COLAB else Path("./data/mvtec_ad")

def has_categories(root, cats):
    return root is not None and all((Path(root) / c / "test").is_dir() for c in cats)

def search_root(base, cats):
    # 在 base 底下找出包含所有類別的資料夾（處理壓縮檔多包一層目錄的情況）
    base = Path(base)
    if has_categories(base, cats):
        return base
    if base.is_dir():
        for p in base.rglob(f"{cats[0]}"):
            if p.is_dir() and has_categories(p.parent, cats):
                return p.parent
    return None

def extract_categories(tar_path, dest, cats):
    # 只解壓需要的類別，省時間與空間
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    print(f"解壓 {tar_path} 中的 {cats} -> {dest}（完整版約需數分鐘）")
    with tarfile.open(tar_path) as tar:
        members = [m for m in tar.getmembers()
                   if any(c in Path(m.name).parts for c in cats)]
        try:
            tar.extractall(dest, members=members, filter="data")
        except TypeError:  # Python < 3.12
            tar.extractall(dest, members=members)
    print(f"完成，共 {len(members)} 個檔案")

def locate_mvtec():
    candidates = [DATA_ROOT, os.environ.get("MVTEC_ROOT"), EXTRACT_TO,
                  "./data/mvtec_ad", "../data/mvtec_ad", "./mvtec_anomaly_detection",
                  "D:/datasets/mvtec_ad", "/data/mvtec_ad", "/content/mvtec_ad"]
    for c in candidates:
        if c and Path(c).exists():
            r = search_root(c, CATEGORIES)
            if r is not None:
                return r
    tar = MVTEC_TAR
    if tar and str(tar).startswith("/content/drive") and IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    if not tar and MVTEC_URL:
        tar = EXTRACT_TO.parent / Path(MVTEC_URL.split("?")[0]).name
        if not Path(tar).exists():
            print("下載中：", MVTEC_URL)
            urllib.request.urlretrieve(MVTEC_URL, tar)
    if tar and Path(tar).exists():
        extract_categories(tar, EXTRACT_TO, CATEGORIES)
        return search_root(EXTRACT_TO, CATEGORIES)
    raise FileNotFoundError(
        "找不到 MVTec AD。請設定 DATA_ROOT（已解壓的資料夾）、MVTEC_TAR（壓縮檔）或 MVTEC_URL，"
        f"並確認包含這些類別：{CATEGORIES}")

DATA_ROOT = locate_mvtec()
print("DATA_ROOT =", DATA_ROOT)
for c in CATEGORIES:
    n_train = len(list((DATA_ROOT / c / "train" / "good").glob("*.png")))
    test_types = sorted(p.name for p in (DATA_ROOT / c / "test").iterdir() if p.is_dir())
    print(f"  {c:12s} official train(good)={n_train:4d} | test types = {test_types}")

## §3 Track A 的資料切分：最重要的一步

官方 train set **只有良品**，監督式分類器學不到「瑕疵長什麼樣子」。期中規定 Track A：

1. 把官方 **test set** 每個類別 **50/50** 切開，並依 **defect type 分層**（每種瑕疵兩邊都有）。
2. 一半當開發集（dev），另一半當 **held-out**，調參期間**完全不看**。
3. 早停與門檻需要驗證資料，所以從 dev 中再切 20% 當 **val**。

```
official test ──┬── 50% dev ──┬── 80% train   ← 訓練
                │             └── 20% val     ← 早停、選門檻
                └── 50% held-out              ← 只在最後報告
```

> **切分 seed（`SPLIT_SEED`）跟訓練 seed（`SEEDS`）是兩回事。**切分固定一次並存成 CSV；3 個訓練 seed 都用同一份切分，這樣 mean ± std 才是在量「訓練的隨機性」。

In [ ]:
from sklearn.model_selection import train_test_split

def build_split(root, category, seed=SPLIT_SEED, heldout_frac=0.5, val_frac=0.2):
    csv = SPLIT_DIR / f"{category}_split_seed{seed}.csv"
    if csv.exists():                       # 已經切過就直接讀，保證每次都一樣
        return pd.read_csv(csv).fillna({"mask": ""})
    rows = []
    for dt_dir in sorted((Path(root) / category / "test").iterdir()):
        if not dt_dir.is_dir():
            continue
        for f in sorted(dt_dir.glob("*.png")):
            m = Path(root) / category / "ground_truth" / dt_dir.name / f"{f.stem}_mask.png"
            rows.append(dict(category=category, path=str(f), defect_type=dt_dir.name,
                             label=int(dt_dir.name != "good"), mask=str(m) if m.exists() else ""))
    df = pd.DataFrame(rows)
    dev, held = train_test_split(df, test_size=heldout_frac, stratify=df["defect_type"], random_state=seed)
    tr, va = train_test_split(dev, test_size=val_frac, stratify=dev["label"], random_state=seed)
    df["split"] = ""
    df.loc[tr.index, "split"] = "train"
    df.loc[va.index, "split"] = "val"
    df.loc[held.index, "split"] = "heldout"
    df.to_csv(csv, index=False)
    return df

SPLITS = {c: build_split(DATA_ROOT, c) for c in CATEGORIES}

for c, df in SPLITS.items():
    print(f"\n=== {c} ===")
    tab = pd.crosstab(df["defect_type"], df["split"], margins=True)
    display(tab[["train", "val", "heldout", "All"]])

In [ ]:
def load_rgb(path, size=None):
    img = Image.open(path).convert("RGB")      # grid / screw / zipper 是灰階，統一轉 RGB
    return img.resize((size, size), Image.BILINEAR) if size else img

def load_mask(path, size=None):
    if not path:
        return None
    m = Image.open(path).convert("L")
    return m.resize((size, size), Image.NEAREST) if size else m

def show_examples(category, n_per_type=1):
    df = SPLITS[category]
    types = sorted(df["defect_type"].unique(), key=lambda t: (t != "good", t))
    fig, axes = plt.subplots(2, len(types), figsize=(2.6 * len(types), 5.4))
    for j, t in enumerate(types):
        r = df[df["defect_type"] == t].iloc[0]
        img = np.asarray(load_rgb(r["path"], CACHE_SIZE))
        axes[0, j].imshow(img); axes[0, j].set_title(t, fontsize=9)
        axes[1, j].imshow(img)
        m = load_mask(r["mask"], CACHE_SIZE)
        if m is not None:
            axes[1, j].imshow(np.ma.masked_where(np.asarray(m) == 0, np.asarray(m)), cmap="autumn", alpha=0.6)
            axes[1, j].set_title(f"mask: {np.asarray(m).mean()/255*100:.2f}% px", fontsize=8)
        for a in axes[:, j]: a.axis("off")
    fig.suptitle(f"{category}: one example per defect type (row 2 = ground-truth mask)")
    plt.tight_layout(); plt.show()

for c in CATEGORIES:
    show_examples(c)

**觀察**：mask 上方的百分比是瑕疵占整張圖的像素比例。很多瑕疵不到 1%，這就是為什麼「隨便裁切、模糊」可能讓瑕疵消失（§9 會量化）。

## §4 Dataset 與 transforms

- 圖片先縮到 `CACHE_SIZE` 並放進記憶體：資料量小，這樣每個 epoch 只要幾秒。
- **一定要用 ImageNet 的 mean/std 做 Normalize**：預訓練權重是在這個分布上學的。
- augmentation 用名稱指定（例如 `"hflip"`、`"hflip+color"`），方便 §9 一次只加一種。

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

_CACHE = {}
def cached(path):
    if path not in _CACHE:
        _CACHE[path] = load_rgb(path, CACHE_SIZE)
    return _CACHE[path]

class RandomRot90:
    # 隨機旋轉 0/90/180/270 度（對 texture 通常安全，對 object 可能改變語意）
    def __call__(self, img):
        k = random.randint(0, 3)
        return img.rotate(90 * k) if k else img
    def __repr__(self):
        return "RandomRot90()"

# 每種 augmentation：(PIL 階段的操作, Tensor 階段的操作)
AUGMENTATIONS = {
    "none":    ([], []),
    "hflip":   ([T.RandomHorizontalFlip()], []),
    "vflip":   ([T.RandomVerticalFlip()], []),
    "rot90":   ([RandomRot90()], []),
    "rotate15":([T.RandomRotation(15)], []),
    "color":   ([T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05)], []),
    "blur":    ([T.GaussianBlur(kernel_size=7, sigma=(0.5, 2.5))], []),
    "crop":    ([T.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0), ratio=(0.9, 1.1))], []),
    "erasing": ([], [T.RandomErasing(p=0.5, scale=(0.02, 0.15))]),
}

def build_transform(aug="none", train=True, size=IMG_SIZE):
    pil_ops, tensor_ops = [], []
    if train:
        for name in aug.split("+"):
            p, t = AUGMENTATIONS[name]
            pil_ops += p; tensor_ops += t
    return T.Compose([T.Resize((size, size)), *pil_ops,
                      T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD), *tensor_ops])

class DefectDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.tf = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.tf(cached(r["path"])), int(r["label"]), i

# 預先把所有圖片載入記憶體
t0 = time.time()
for df in SPLITS.values():
    for p in df["path"]:
        cached(p)
print(f"cached {len(_CACHE)} images in {time.time()-t0:.1f}s")
print(build_transform("hflip+erasing"))

## §5 打開預訓練 CNN 看看（Topic 10 回顧）

在開始訓練之前，先看預訓練模型「已經會什麼」。

### 5.1 建立模型：把最後一層換掉

Transfer learning 的標準做法：保留 backbone（卷積層），把原本輸出 1000 類 ImageNet 的分類層換成輸出 2 類（良品 / 瑕疵）的新層。不同架構的「最後一層」名字不同：

| 架構 | 最後一層 | 參數量 |
|---|---|---|
| ResNet-18 / 50 | `model.fc` | 11.7M / 25.6M |
| MobileNetV3-Large | `model.classifier[-1]` | 5.5M |
| EfficientNet-B0 | `model.classifier[-1]` | 5.3M |
| VGG16 | `model.classifier[-1]` | 138M |

In [ ]:
WEIGHTS = {
    "resnet18":          models.ResNet18_Weights.IMAGENET1K_V1,
    "resnet50":          models.ResNet50_Weights.IMAGENET1K_V2,
    "mobilenet_v3_large":models.MobileNet_V3_Large_Weights.IMAGENET1K_V2,
    "efficientnet_b0":   models.EfficientNet_B0_Weights.IMAGENET1K_V1,
    "vgg16":             models.VGG16_Weights.IMAGENET1K_V1,
}

def build_model(arch="resnet18", pretrained=True, num_classes=2):
    # 回傳 (model, head)：head 是新換上去、要從零學的分類層所在模組
    weights = WEIGHTS[arch] if pretrained else None
    m = getattr(models, arch)(weights=weights)
    if arch.startswith("resnet"):
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        head = m.fc
    else:  # mobilenet / efficientnet / vgg 的分類器都在 m.classifier，最後一個是 Linear
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
        head = m.classifier
    return m, head

def count_params(m, trainable_only=False):
    return sum(p.numel() for p in m.parameters() if p.requires_grad or not trainable_only)

model, head = build_model("resnet18", pretrained=True)
print(model.fc)
print(f"total params: {count_params(model)/1e6:.2f}M | head params: {sum(p.numel() for p in head.parameters()):,}")

### 5.2 第一層 filter：預訓練 vs 隨機初始化

ImageNet 預訓練的第一層學到的是**邊緣、顏色、紋理**偵測器，這些對刮痕、裂縫、污漬都很有用。隨機初始化的 filter 則只是雜訊，要從幾十張圖重新學起。

In [ ]:
def show_conv1_filters(m, title, ax_row):
    w = m.conv1.weight.detach().cpu()[:16]            # 前 16 個 7x7x3 filter
    w = (w - w.amin(dim=(1, 2, 3), keepdim=True)) / (w.amax(dim=(1, 2, 3), keepdim=True) - w.amin(dim=(1, 2, 3), keepdim=True) + 1e-8)
    for k, a in enumerate(ax_row):
        a.imshow(w[k].permute(1, 2, 0)); a.axis("off")
    ax_row[0].set_title(title, loc="left", fontsize=10)

fig, axes = plt.subplots(2, 16, figsize=(16, 2.4))
show_conv1_filters(build_model("resnet18", True)[0],  "ImageNet-pretrained conv1", axes[0])
show_conv1_filters(build_model("resnet18", False)[0], "random init conv1", axes[1])
plt.tight_layout(); plt.show()

### 5.3 各層 feature map：越深越抽象

拿一張瑕疵圖丟進預訓練 ResNet-18，看 `layer1`～`layer4` 的輸出（每層取通道平均）。淺層保留細節與邊緣，深層解析度變低、語意變強。**想一想：細小的刮痕在 `layer4`（7×7）還看得到嗎？**

In [ ]:
def feature_maps(m, x):
    feats, hooks = {}, []
    for name in ["layer1", "layer2", "layer3", "layer4"]:
        hooks.append(getattr(m, name).register_forward_hook(
            lambda mod, inp, out, n=name: feats.__setitem__(n, out.detach().cpu())))
    with torch.no_grad():
        m(x)
    for h in hooks: h.remove()
    return feats

m_pre = build_model("resnet18", True)[0].eval()
tf_eval = build_transform(train=False)
fig, axes = plt.subplots(len(CATEGORIES), 6, figsize=(15, 2.8 * len(CATEGORIES)), squeeze=False)
for i, c in enumerate(CATEGORIES):
    r = SPLITS[c].query("label == 1 and mask != ''").iloc[0]
    x = tf_eval(cached(r["path"])).unsqueeze(0)
    fm = feature_maps(m_pre, x)
    axes[i, 0].imshow(cached(r["path"])); axes[i, 0].set_title(f"{c}/{r['defect_type']}", fontsize=9)
    axes[i, 1].imshow(load_mask(r["mask"], CACHE_SIZE), cmap="gray"); axes[i, 1].set_title("mask", fontsize=9)
    for j, (n, f) in enumerate(fm.items()):
        axes[i, j + 2].imshow(f[0].mean(0), cmap="viridis")
        axes[i, j + 2].set_title(f"{n} {tuple(f.shape[1:])}", fontsize=9)
    for a in axes[i]: a.axis("off")
plt.tight_layout(); plt.show()

### 5.4 為什麼 Transfer Learning 有效？ImageNet 特徵的分布

**還沒有任何訓練**，直接把預訓練 ResNet-18 當特徵抽取器（拿掉 `fc`，得到 512 維向量），用 PCA 投影到 2 維。如果良品與瑕疵已經有一點分開，代表 ImageNet 學到的特徵對這個任務本來就有用 → 只需要少量資料就能學會。

In [ ]:
from sklearn.decomposition import PCA

@torch.no_grad()
def extract_features(m, df, bs=32):
    m = m.to(DEVICE).eval()
    dl = DataLoader(DefectDataset(df, build_transform(train=False)), batch_size=bs, shuffle=False)
    return torch.cat([m(x.to(DEVICE)).cpu() for x, _, _ in dl]).numpy()

feat_net = build_model("resnet18", True)[0]
feat_net.fc = nn.Identity()

fig, axes = plt.subplots(1, len(CATEGORIES), figsize=(6 * len(CATEGORIES), 4.6), squeeze=False)
for a, c in zip(axes[0], CATEGORIES):
    df = SPLITS[c]
    z = PCA(n_components=2, random_state=0).fit_transform(extract_features(feat_net, df))
    for t in sorted(df["defect_type"].unique(), key=lambda t: (t != "good", t)):
        k = (df["defect_type"] == t).values
        a.scatter(z[k, 0], z[k, 1], s=22, label=t,
                  marker="o" if t == "good" else "^", edgecolor="k" if t == "good" else None, linewidth=0.5)
    a.set_title(f"{c}: frozen ImageNet ResNet-18 features (PCA)"); a.legend(fontsize=7)
plt.tight_layout(); plt.show()

## §6 訓練與評估函式

### 三種 transfer 策略（Topic 10 的核心）

| 策略 | 預訓練權重 | 哪些參數會更新 | 學習率 | 何時用 |
|---|---|---|---|---|
| `scratch` | ✗ | 全部 | 1e-3 | 對照組：資料少時通常最差 |
| `frozen` | ✓ | 只有新的分類層 | 1e-3 | 資料**非常少**、或任務與 ImageNet 很像 |
| `finetune` | ✓ | 全部 | 1e-4（投影片建議） | 資料稍多、或任務與 ImageNet 差異大 |
| `discriminative` | ✓ | 全部，但分層設定 lr | backbone 1e-4 / head 1e-3 | 新的 head 學快一點、舊的特徵慢慢調 |

> ⚠️ **凍結時的陷阱：BatchNorm。**`requires_grad=False` 只會停止權重更新，但 `model.train()` 時 BatchNorm 的 running mean/var 仍然會被小 batch 的統計量覆蓋，等於偷偷破壞了預訓練特徵。所以 `frozen` 模式要把被凍結的 BN 層維持在 `eval()`。

### 評估指標（期中必須回報）
- **Image-level AUROC**：主要分數，與門檻無關、不受類別不平衡影響。用 **瑕疵機率**（softmax 第 1 類）計算，不是 0/1 預測。
- **Recall @ 5% FPR**：只允許 5% 良品被誤判時，能抓到多少比例的瑕疵。
- **混淆矩陣**：門檻在 **val** 上決定（不可以用 held-out 挑門檻），再套到 held-out。

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

def recall_at_fpr(y, s, target=TARGET_FPR):
    fpr, tpr, _ = roc_curve(y, s)
    return float(tpr[fpr <= target].max())

def pick_threshold(y_val, s_val, target=TARGET_FPR):
    # 在 val 上找「良品誤判率 <= target」的最低門檻（最大化 recall）
    # 注意：val 裡的良品很少，FPR 只能取很粗的值 → 報告中應該說明這個限制
    s_good = np.sort(np.asarray(s_val)[np.asarray(y_val) == 0])
    if len(s_good) == 0:
        return 0.5
    k = int(np.floor(target * len(s_good)))          # 最多允許 k 張良品超過門檻
    thr = s_good[len(s_good) - 1 - k] if k < len(s_good) else s_good[0]
    return float(np.nextafter(thr, 1.0))             # 嚴格大於該良品分數

def compute_metrics(y, s, thr):
    y, s = np.asarray(y), np.asarray(s)
    pred = (s >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(auroc=roc_auc_score(y, s), recall_at_5fpr=recall_at_fpr(y, s),
                threshold=thr, tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
                recall=float(tp / max(tp + fn, 1)), false_alarm_rate=float(fp / max(fp + tn, 1)))

In [ ]:
MODES = {
    #                pretrained, train_backbone, backbone_lr, head_lr
    "scratch":        (False, True,  1e-3, 1e-3),
    "frozen":         (True,  False, 0.0,  1e-3),
    "finetune":       (True,  True,  1e-4, 1e-4),
    "discriminative": (True,  True,  1e-4, 1e-3),
}

def make_loader(df, aug, train, seed):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(DefectDataset(df, build_transform(aug, train=train)),
                      batch_size=BATCH_SIZE, shuffle=train, generator=g if train else None,
                      num_workers=NUM_WORKERS, drop_last=False)

def keep_frozen_bn_in_eval(model):
    for mod in model.modules():
        if isinstance(mod, nn.modules.batchnorm._BatchNorm) and not any(p.requires_grad for p in mod.parameters()):
            mod.eval()

@torch.no_grad()
def predict(model, loader):
    model.eval()
    ys, ss, idx, loss_sum = [], [], [], 0.0
    for x, y, i in loader:
        logits = model(x.to(DEVICE))
        loss_sum += F.cross_entropy(logits, y.to(DEVICE), reduction="sum").item()
        ss.append(torch.softmax(logits, 1)[:, 1].cpu()); ys.append(y); idx.append(i)
    return torch.cat(ys).numpy(), torch.cat(ss).numpy(), torch.cat(idx).numpy(), loss_sum / len(loader.dataset)

def run_experiment(category, mode="finetune", arch="resnet18", aug="none", seed=0,
                   weight_decay=0.05, epochs=EPOCHS, patience=PATIENCE, verbose=False, keep_model=False):
    set_seed(seed)
    df = SPLITS[category]
    tr, va, ho = (df[df.split == s] for s in ("train", "val", "heldout"))
    dl_tr, dl_va, dl_ho = make_loader(tr, aug, True, seed), make_loader(va, aug, False, seed), make_loader(ho, aug, False, seed)

    pretrained, train_backbone, lr_bb, lr_head = MODES[mode]
    model, head = build_model(arch, pretrained)
    head_ids = {id(p) for p in head.parameters()}
    for p in model.parameters():
        p.requires_grad = train_backbone or id(p) in head_ids
    model.to(DEVICE)

    groups = [{"params": list(head.parameters()), "lr": lr_head}]
    bb = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    if bb:
        groups.append({"params": bb, "lr": lr_bb})
    opt = torch.optim.AdamW(groups, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    # 類別權重：瑕疵/良品數量不平衡時，讓兩類對 loss 的貢獻相當
    counts = np.bincount(tr["label"], minlength=2)
    class_w = torch.tensor(len(tr) / (2 * np.maximum(counts, 1)), dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_w)

    best_loss, best_state, best_ep, bad, hist = float("inf"), None, 0, 0, []
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        if not train_backbone:
            keep_frozen_bn_in_eval(model)
        tl = 0.0
        for x, y, _ in dl_tr:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = criterion(model(x), y)
            loss.backward(); opt.step()
            tl += loss.item() * len(y)
        sched.step()
        yv, sv, _, vl = predict(model, dl_va)
        vauc = roc_auc_score(yv, sv) if len(set(yv)) > 1 else float("nan")
        hist.append(dict(epoch=ep, train_loss=tl / len(tr), val_loss=vl, val_auroc=vauc))
        if verbose:
            print(f"  ep {ep:2d} train_loss {tl/len(tr):.4f} val_loss {vl:.4f} val_auroc {vauc:.3f}")
        if vl < best_loss - 1e-4:
            best_loss, best_ep, bad = vl, ep, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)

    yv, sv, _, _ = predict(model, dl_va)
    thr = pick_threshold(yv, sv)
    yh, sh, ih, _ = predict(model, dl_ho)                 # held-out：只在這裡用一次
    res = dict(category=category, mode=mode, arch=arch, aug=aug, seed=seed,
               best_epoch=best_ep, secs=round(time.time() - t0, 1), **compute_metrics(yh, sh, thr))
    scores = ho.iloc[ih].assign(score=sh)
    return res, pd.DataFrame(hist), scores, (model.cpu() if keep_model else None)

In [ ]:
# 先用一個設定跑一次，確認整條流程沒問題（看每個 epoch 的變化）
res, hist, scores, _ = run_experiment(CATEGORIES[0], mode="finetune", seed=0, verbose=True)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in res.items()})

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
hist.plot(x="epoch", y=["train_loss", "val_loss"], ax=ax[0], title="loss")
hist.plot(x="epoch", y="val_auroc", ax=ax[1], title="val AUROC", ylim=(0, 1.02))
plt.tight_layout(); plt.show()

**討論**：val 只有十幾張圖，所以 val loss 與 val AUROC 會跳動。這是小資料的現實，也是為什麼期中要求 **3 個 seed 的 mean ± std**：單一 seed 的結果可能只是運氣。

## §7 實驗一：Scratch vs Frozen vs Fine-tune

這是期中 pipeline 的 **1. Baseline** 與 **2. Transfer learning**。為了公平比較，以下設定**全部固定**：ResNet-18、無 augmentation、相同切分、相同 epoch/早停規則，只改「訓練策略」。

In [ ]:
ALL_RESULTS, ALL_SCORES, MODELS = [], {}, {}

def run_grid(configs, tag, keep=False):
    rows = []
    for cfg in configs:
        for c in CATEGORIES:
            for s in SEEDS:
                res, _, sc, m = run_experiment(c, seed=s, keep_model=keep, **cfg)
                res["experiment"] = tag
                rows.append(res)
                key = (tag, c, cfg.get("mode", "finetune"), cfg.get("arch", "resnet18"), cfg.get("aug", "none"), s)
                ALL_SCORES[key] = sc
                if keep: MODELS[key] = m
                print(f"[{tag}] {c:11s} {str(cfg):55s} seed={s}  AUROC={res['auroc']:.3f}  "
                      f"R@5%FPR={res['recall_at_5fpr']:.3f}  ep={res['best_epoch']:2d}  {res['secs']}s")
    ALL_RESULTS.extend(rows)
    return pd.DataFrame(rows)

def summarize(df, by):
    # 每個類別：mean ± std over seeds；再加上「類別平均」那一欄
    fmt = lambda g: f"{g.mean():.3f} ± {g.std(ddof=0):.3f}" if len(g) > 1 else f"{g.mean():.3f}"
    per_cat = df.groupby(by + ["category"])["auroc"].apply(fmt).unstack("category")
    mean_cat = df.groupby(by + ["seed"])["auroc"].mean().groupby(by).apply(fmt).rename("mean (cats)")
    rec = df.groupby(by + ["seed"])["recall_at_5fpr"].mean().groupby(by).apply(fmt).rename("R@5%FPR (mean)")
    return pd.concat([per_cat, mean_cat, rec], axis=1)

exp1 = run_grid([dict(mode=m) for m in ["scratch", "frozen", "finetune", "discriminative"]], "exp1_strategy", keep=True)
display(summarize(exp1, ["mode"]))

In [ ]:
def bar_by(df, by, title):
    g = df.groupby([by, "category"])["auroc"].agg(["mean", "std"]).reset_index()
    order = list(dict.fromkeys(df[by]))
    fig, ax = plt.subplots(figsize=(1.6 * len(order) + 3, 3.6))
    w = 0.8 / len(CATEGORIES)
    for k, c in enumerate(CATEGORIES):
        gc = g[g.category == c].set_index(by).loc[order]
        ax.bar(np.arange(len(order)) + k * w, gc["mean"], w, yerr=gc["std"].fillna(0), capsize=3, label=c)
    ax.set_xticks(np.arange(len(order)) + w * (len(CATEGORIES) - 1) / 2); ax.set_xticklabels(order)
    ax.axhline(0.5, ls="--", c="gray", lw=1); ax.set_ylim(0.3, 1.02); ax.set_ylabel("held-out AUROC")
    ax.set_title(title); ax.legend(); plt.tight_layout(); plt.show()

bar_by(exp1, "mode", "Experiment 1: training strategy (ResNet-18, no augmentation)")

### 🗣️ 課堂討論
1. `scratch` 比 `frozen` 差多少？用 §5.2 的 filter 圖解釋原因。
2. `frozen` 只訓練 1,026 個參數，為什麼有時候跟 `finetune`（1,100 萬個參數）差不多？什麼情況下 `finetune` 會明顯比較好？（提示：§5.4 的 PCA 圖中，良品與瑕疵分得開嗎？）
3. texture 與 object 類別的差距一樣嗎？

## §8 實驗二：換 Backbone（VGG16 / MobileNet / ResNet / EfficientNet）

對應 Topic 10 影片 *Transfer learning – explained*。為了在課堂時間內跑完，這裡用 **`frozen`** 模式（只換 backbone，其他全部相同）。

> 💡 時間足夠的組別可以把 `BACKBONE_MODE` 改成 `"finetune"` 再跑一次，比較兩種模式下的排名是否改變。VGG16 有 1.38 億個參數，fine-tune 時注意顯示記憶體（ST301 的 24 GB 沒問題；Colab T4 可把 `BATCH_SIZE` 調小）。

In [ ]:
BACKBONES = ["resnet18", "mobilenet_v3_large", "efficientnet_b0", "vgg16"]
BACKBONE_MODE = "frozen"

info = []
for a in BACKBONES:
    m, h = build_model(a, pretrained=False)
    info.append(dict(arch=a, params_M=round(count_params(m) / 1e6, 1),
                     head_params=sum(p.numel() for p in h.parameters())))
display(pd.DataFrame(info).set_index("arch"))

exp2 = run_grid([dict(mode=BACKBONE_MODE, arch=a) for a in BACKBONES], "exp2_backbone")
display(summarize(exp2, ["arch"]))
bar_by(exp2, "arch", f"Experiment 2: backbone comparison ({BACKBONE_MODE})")

**討論**：參數量最大的模型一定最好嗎？在產線上，模型還需要考慮推論速度與部署硬體，什麼情況你會選 MobileNet？

> 注意 VGG16 在 `frozen` 模式下，`classifier` 裡兩層 4096 維的全連接層也會被訓練（我們把整個 `classifier` 當作 head），所以它的可訓練參數遠多於其他模型，這個比較並不完全公平。報告中要能指出這種差異。

## §9 Augmentation：哪些會讓標籤「說謊」？

Augmentation 的前提是：**轉換後的圖片，標籤仍然正確**。在瑕疵檢測中，這個前提經常被打破：

| Augmentation | 風險 | 要找的例子 |
|---|---|---|
| Random crop | 把瑕疵裁掉 → 「瑕疵」標籤的圖看起來是良品 | 邊緣的小刮痕 |
| Color jitter | 掩蓋或假造顏色類瑕疵 | 地毯、皮革上的污漬 |
| Blur | 抹除細微瑕疵 | 細裂縫 |
| Rotation / flip | texture 通常安全；object 可能改變意義 | transistor 放錯位置 |
| Random erasing | 擦掉瑕疵，或自己製造一個「洞」 | hazelnut、pill 表面 |

### 9.1 先用眼睛看

In [ ]:
def aug_gallery(category, augs=("none", "hflip", "rot90", "color", "blur", "crop", "erasing"), n_draw=1, seed=3):
    r = SPLITS[category].query("label == 1 and mask != ''").iloc[0]
    img = cached(r["path"])
    fig, axes = plt.subplots(1, len(augs), figsize=(2.3 * len(augs), 2.6))
    for a, name in zip(axes, augs):
        set_seed(seed)
        x = build_transform(name, train=True)(img)
        x = (x * torch.tensor(IMAGENET_STD)[:, None, None] + torch.tensor(IMAGENET_MEAN)[:, None, None]).clamp(0, 1)
        a.imshow(x.permute(1, 2, 0)); a.set_title(name, fontsize=9); a.axis("off")
    fig.suptitle(f"{category}/{r['defect_type']} (label = defect)", fontsize=10)
    plt.tight_layout(); plt.show()

for c in CATEGORIES:
    aug_gallery(c)

### 9.2 用 mask 量化：Random crop 會把多少瑕疵裁掉？

只看圖很主觀。MVTec AD 有像素級的 ground-truth mask，所以我們可以**把圖片和 mask 一起做同樣的裁切**，計算裁切後還剩多少瑕疵像素。如果剩下不到 10%，這張圖仍被標為「瑕疵」，但模型幾乎看不到瑕疵 → **標籤失真**。

做法：用 `T.RandomResizedCrop.get_params` 抽出一組隨機裁切位置 `(i, j, h, w)`，把**同一組**參數套在影像與 mask 上。

In [ ]:
import torchvision.transforms.functional as TF

def crop_retention(category, scale=(0.4, 1.0), ratio=(0.9, 1.1), trials=50, seed=0):
    # 用「同一組」隨機裁切參數 (i, j, h, w) 同時裁圖片與 mask，計算裁切後保留的瑕疵像素比例
    torch.manual_seed(seed)
    rows, worst = [], None
    for _, r in SPLITS[category].query("label == 1 and mask != ''").iterrows():
        img = cached(r["path"])
        m = torch.from_numpy(np.asarray(load_mask(r["mask"], CACHE_SIZE)) > 0)
        total = m.sum().item()
        if total == 0:
            continue
        for _ in range(trials):
            i, j, h, w = T.RandomResizedCrop.get_params(img, scale=scale, ratio=ratio)
            ret = m[i:i + h, j:j + w].sum().item() / total
            rows.append(dict(defect_type=r["defect_type"], mask_pct=total / m.numel() * 100, retained=ret))
            if worst is None or ret < worst[0] or (ret == worst[0] and total > worst[3]):
                worst = (ret, r, (i, j, h, w), total)
    return pd.DataFrame(rows), worst

for c in CATEGORIES:
    d, worst = crop_retention(c)
    lost = (d["retained"] < 0.10).mean() * 100
    print(f"\n{c}: RandomResizedCrop(scale=(0.4,1.0)) 後，{lost:.1f}% 的瑕疵樣本只剩不到 10% 的瑕疵像素")
    display(d.assign(lost=d.retained < 0.10).groupby("defect_type")[["mask_pct", "lost"]].mean()
             .rename(columns={"mask_pct": "avg defect area %", "lost": "P(defect cropped out)"}).round(3))

    ret, r, (i, j, h, w), total = worst
    img = cached(r["path"]); m = load_mask(r["mask"], CACHE_SIZE)
    fig, ax = plt.subplots(1, 3, figsize=(9, 3.2))
    ax[0].imshow(img); ax[0].imshow(np.ma.masked_equal(np.asarray(m), 0), cmap="autumn", alpha=0.6)
    ax[0].add_patch(plt.Rectangle((j, i), w, h, fill=False, ec="cyan", lw=2))
    ax[0].set_title(f"original ({r['defect_type']}), cyan = crop\ndefect area {total/np.asarray(m).size*100:.2f}%", fontsize=9)
    ax[1].imshow(TF.resized_crop(img, i, j, h, w, [IMG_SIZE, IMG_SIZE])); ax[1].set_title("after random crop\nlabel still = DEFECT", fontsize=9)
    ax[2].imshow(TF.resized_crop(m, i, j, h, w, [IMG_SIZE, IMG_SIZE], interpolation=T.InterpolationMode.NEAREST), cmap="gray", vmin=0, vmax=255)
    ax[2].set_title(f"mask after crop\nretained {ret*100:.0f}% of defect pixels", fontsize=9)
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()

### 9.3 幾何轉換對 object 類別：旋轉會不會「製造」瑕疵？

`transistor` 有一種瑕疵叫 `misplaced`（元件放錯位置或缺件）。對**良品** transistor 做旋轉，看起來是不是很像 misplaced？如果是，那麼「旋轉後的良品」其實應該被標成瑕疵，標籤就錯了。

（若你的類別不含 transistor，這格會自動改用第一個 object 類別的良品示範。）

In [ ]:
OBJECTS = ["bottle", "cable", "capsule", "hazelnut", "metal_nut", "pill", "screw", "toothbrush", "transistor", "zipper"]
obj = "transistor" if "transistor" in CATEGORIES else next((c for c in CATEGORIES if c in OBJECTS), None)
tex = next((c for c in CATEGORIES if c not in OBJECTS), None)

if obj:
    df = SPLITS[obj]
    good = cached(df.query("defect_type == 'good'").iloc[0]["path"])
    panels = [("good", good), ("good, rot 90°", good.rotate(90)), ("good, rot 180°", good.rotate(180)),
              ("good, vflip", good.transpose(Image.FLIP_TOP_BOTTOM))]
    if "misplaced" in set(df.defect_type):
        for _, r in df.query("defect_type == 'misplaced'").head(2).iterrows():
            panels.append(("REAL defect: misplaced", cached(r["path"])))
    fig, ax = plt.subplots(1, len(panels), figsize=(2.4 * len(panels), 2.8))
    for a, (t, im) in zip(ax, panels):
        a.imshow(im); a.set_title(t, fontsize=9, color="red" if "REAL" in t else "black"); a.axis("off")
    fig.suptitle(f"{obj}: geometric augmentation on a GOOD part", fontsize=10); plt.tight_layout(); plt.show()

if tex:
    good = cached(SPLITS[tex].query("defect_type == 'good'").iloc[0]["path"])
    fig, ax = plt.subplots(1, 4, figsize=(9.6, 2.8))
    for a, (t, im) in zip(ax, [("good", good), ("rot 90°", good.rotate(90)), ("rot 180°", good.rotate(180)), ("hflip", good.transpose(Image.FLIP_LEFT_RIGHT))]):
        a.imshow(im); a.set_title(t, fontsize=9); a.axis("off")
    fig.suptitle(f"{tex} (texture): still looks normal after rotation/flip?", fontsize=10); plt.tight_layout(); plt.show()

### 9.4 一次只加一種 augmentation

期中最低要求：**至少測試 4 種 augmentation**，並展示**至少一張** augmentation 讓標籤失真的圖（§9.2 / §9.3 就是範例）。

以下固定 ResNet-18 + `finetune`，**一次只加一種** augmentation，與 `none` 比較。

In [ ]:
AUG_LIST = ["none", "hflip", "rot90", "color", "blur", "crop", "erasing"]
exp3 = run_grid([dict(mode="finetune", aug=a) for a in AUG_LIST], "exp3_augmentation")
display(summarize(exp3, ["aug"]))
bar_by(exp3, "aug", "Experiment 3: one augmentation at a time (ResNet-18, finetune)")

# 相對於 none 的變化量（每個類別分開看）
base = exp3[exp3.aug == "none"].groupby("category")["auroc"].mean()
delta = exp3.groupby(["aug", "category"])["auroc"].mean().unstack() - base
display(delta.loc[AUG_LIST].style.format("{:+.3f}").background_gradient(cmap="RdYlGn", vmin=-0.1, vmax=0.1, axis=None))

### 🗣️ 課堂討論
- 哪些 augmentation 對 texture 有幫助、對 object 沒幫助（或反過來）？跟 §9.3 的觀察一致嗎？
- `crop` 的結果和 §9.2 算出的「瑕疵被裁掉比例」有關聯嗎？
- 只有 1 個 seed 時，±0.01 的差距可信嗎？（這就是為什麼期中要 3 個 seed）

> **報告寫法提示**：不要只寫「blur 讓 AUROC 下降 0.03」，要寫「blur 讓 AUROC 下降 0.03，**因為**…（附圖：模糊後的細裂縫）」。投影片說得很清楚：**解釋 why，不只是 what**。

## §10 門檻、ROC 與混淆矩陣

AUROC 不需要門檻，但產線上必須做出「放行 / 攔下」的決定。門檻在 **val** 上決定（目標：良品誤判率 ≤ 5%），然後套用到 held-out。

In [ ]:
def plot_roc_cm(key):
    sc = ALL_SCORES[key]
    res = next(r for r in ALL_RESULTS if (r["experiment"], r["category"], r["mode"], r["arch"], r["aug"], r["seed"]) == key)
    fpr, tpr, _ = roc_curve(sc.label, sc.score)
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
    ax[0].plot(fpr, tpr, lw=2); ax[0].plot([0, 1], [0, 1], "--", c="gray")
    ax[0].axvline(TARGET_FPR, c="r", ls=":", label=f"FPR={TARGET_FPR}")
    ax[0].set_title(f"ROC  AUROC={res['auroc']:.3f}"); ax[0].set_xlabel("false alarm rate"); ax[0].set_ylabel("recall"); ax[0].legend()
    for lab, nm in [(0, "good"), (1, "defect")]:
        ax[1].hist(sc.score[sc.label == lab], bins=20, range=(0, 1), alpha=0.6, label=nm)
    ax[1].axvline(res["threshold"], c="k", ls="--", label=f"thr={res['threshold']:.3f} (from val)")
    ax[1].set_title("held-out score distribution"); ax[1].legend(fontsize=8)
    cm = np.array([[res["tn"], res["fp"]], [res["fn"], res["tp"]]])
    ax[2].imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax[2].text(j, i, int(v), ha="center", va="center", fontsize=14)
    ax[2].set_xticks([0, 1], ["pred OK", "pred defect"]); ax[2].set_yticks([0, 1], ["actual OK", "actual defect"])
    ax[2].set_title(f"false alarms={res['fp']} | missed defects={res['fn']}")
    fig.suptitle(f"{key[1]} · {key[2]} · {key[3]} · aug={key[4]} · seed={key[5]}"); plt.tight_layout(); plt.show()

for c in CATEGORIES:
    plot_roc_cm(("exp1_strategy", c, "finetune", "resnet18", "none", SEEDS[0]))

**討論**：如果漏檢（missed defect）的成本是誤報（false alarm）的 10 倍，你會把門檻往哪邊移？報告中必須用一句話說明你選擇門檻的理由。

## §11 錯誤分析：最糟的 False Negative 與 False Positive

期中報告第 6 節：**3 個 false negative + 3 個 false positive，並解釋原因**。

- **最糟的 FN**：瑕疵圖中分數最低的（模型最有把握說是良品，但其實是瑕疵）
- **最糟的 FP**：良品圖中分數最高的

搭配 **Grad-CAM** 看模型在看哪裡：如果熱圖落在瑕疵上 → 模型學對了；如果落在背景、邊緣或某個與瑕疵無關的位置 → 可能學到了捷徑（shortcut）。

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.act, self.grad = model.eval(), None, None
        layer.register_forward_hook(self._hook)
    def _hook(self, mod, inp, out):
        self.act = out
        out.register_hook(lambda g: setattr(self, "grad", g))
    def __call__(self, x, cls=1):
        x = x.clone().requires_grad_(True)            # 即使 backbone 凍結，也讓梯度能流回來
        self.model.zero_grad()
        self.model(x)[0, cls].backward()
        w = self.grad.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.act).sum(1))[0].detach()
        cam = F.interpolate(cam[None, None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        return ((cam - cam.min()) / (cam.max() - cam.min() + 1e-8)).cpu().numpy()

def error_analysis(key, k=3):
    model = MODELS[key].to(DEVICE)
    cam = GradCAM(model, model.layer4)
    sc = ALL_SCORES[key]
    thr = next(r for r in ALL_RESULTS if (r["experiment"], r["category"], r["mode"], r["arch"], r["aug"], r["seed"]) == key)["threshold"]
    fn = sc[sc.label == 1].nsmallest(k, "score").assign(kind="FN (missed defect)")
    fp = sc[sc.label == 0].nlargest(k, "score").assign(kind="FP (false alarm)")
    cases = pd.concat([fn, fp])
    tf = build_transform(train=False)
    fig, axes = plt.subplots(3, len(cases), figsize=(2.5 * len(cases), 7.6))
    for j, (_, r) in enumerate(cases.iterrows()):
        img = cached(r["path"]).resize((IMG_SIZE, IMG_SIZE))
        h = cam(tf(cached(r["path"])).unsqueeze(0).to(DEVICE))
        wrong = (r["score"] >= thr) != bool(r["label"])
        axes[0, j].imshow(img)
        axes[0, j].set_title(f"{r['kind'].split()[0]} · {r['defect_type']}\nscore={r['score']:.3f}{'  [WRONG]' if wrong else ''}", fontsize=8,
                             color="red" if wrong else "black")
        if r["mask"]:
            axes[1, j].imshow(img); axes[1, j].imshow(np.ma.masked_equal(np.asarray(load_mask(r["mask"], IMG_SIZE)), 0), cmap="autumn", alpha=0.6)
            axes[1, j].set_title("ground truth", fontsize=8)
        else:
            axes[1, j].imshow(img); axes[1, j].set_title("(good: no mask)", fontsize=8)
        axes[2, j].imshow(img); axes[2, j].imshow(h, cmap="jet", alpha=0.45); axes[2, j].set_title("Grad-CAM (defect)", fontsize=8)
        for a in axes[:, j]: a.axis("off")
    fig.suptitle(f"{key[1]}: worst {k} FN and worst {k} FP on held-out (threshold={thr:.3f}; [WRONG] = misclassified at threshold)", fontsize=10)
    plt.tight_layout(); plt.show()
    model.cpu()
    return cases[["kind", "defect_type", "score", "path"]]

for c in CATEGORIES:
    display(error_analysis(("exp1_strategy", c, "finetune", "resnet18", "none", SEEDS[0])))

### 🗣️ 錯誤分析的提問清單（寫報告時逐一回答）
1. 被漏掉的瑕疵有什麼共通點？面積很小？顏色接近背景？位在邊緣？
2. 某一種 defect type 特別容易被漏掉嗎？為什麼？（例如 `tile/gray_stroke` 對比度低）
3. 誤報的良品有什麼特徵？光線、反光、位置偏移、本身就有「看起來像瑕疵」的紋理？
4. Grad-CAM 的高亮區域和 ground-truth mask 重疊嗎？
5. 如果有其中一張你自己也判斷不出來，就照實寫，這也是一個重要發現（可能是標註本身的模糊性）。

> 注意：在只有一半 test set 的情況下，某些類別的 held-out 可能完全沒有分錯（AUROC = 1）。這時仍然要列出**分數最接近門檻**的 3 個 FN 與 3 個 FP（上圖就是這樣排序），並解釋為什麼它們最「危險」。

## §12 儲存所有紀錄（Reproducibility，15%）

期中要求「程式能依 README 重跑、seed 與設定都有紀錄」。以下把結果、設定、環境資訊、切分檔都寫到 `outputs_topic10/`。若在 Git repo 中執行，`config.json` 會一併記錄目前的 commit hash（見 Lab 0 · Git）；結尾帶 `-dirty` 代表有尚未 commit 的修改，正式結果請先 commit 再跑。

In [ ]:
results = pd.DataFrame(ALL_RESULTS)
results.to_csv(OUT_DIR / "results_all_runs.csv", index=False)

import subprocess
def git_version():
    # 在 Git repo 中執行時記錄 commit hash；有未 commit 的修改會加上 -dirty
    try:
        commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL).decode().strip()
        dirty = subprocess.call(["git", "diff", "--quiet"], stderr=subprocess.DEVNULL) != 0
        return commit + ("-dirty" if dirty else "")
    except Exception:
        return "not-a-git-repo"

config = dict(git_commit=git_version(), CATEGORIES=CATEGORIES, IMG_SIZE=IMG_SIZE, CACHE_SIZE=CACHE_SIZE, BATCH_SIZE=BATCH_SIZE,
              EPOCHS=EPOCHS, PATIENCE=PATIENCE, SPLIT_SEED=SPLIT_SEED, SEEDS=SEEDS, TARGET_FPR=TARGET_FPR,
              MODES={k: dict(pretrained=v[0], train_backbone=v[1], backbone_lr=v[2], head_lr=v[3]) for k, v in MODES.items()},
              optimizer="AdamW(weight_decay=0.05)", scheduler="CosineAnnealingLR", early_stopping="val loss",
              env=dict(python=platform.python_version(), torch=torch.__version__, torchvision=torchvision.__version__,
                       device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"))
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False))

print("saved:")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file(): print("  ", p)
display(results.groupby(["experiment", "mode", "arch", "aug"])["auroc"].agg(["mean", "std", "count"]).round(3))

## §12b 把結果 push 到 GitHub（cmd）

在 notebook 外面的 **cmd 視窗**操作（Jupyter 可以繼續開著，另開一個 cmd）。ST301 的電腦可能會還原，**離開教室前一定要 push**。

1. JupyterLab：`Kernel → Restart Kernel and Clear Outputs of All Cells`，然後存檔（notebook 不帶輸出，避免 repo 變大、也減少衝突）。
2. 在 cmd：

```bat
D:
cd \cise2181\cise2181-starter
git status
git add notebooks
git commit -m "Run Topic 10 lab: scratch vs frozen vs fine-tune"
git push
git log --oneline -3
```

- `git status`：看哪些檔案改了。
- `git add`：把要記錄的檔案放進 staging area（index）。
- `git commit`：存成一個有說明的快照；`config.json` 會記下這個 commit 的 hash。
- `git push`：上傳到課程 GitHub organization。

更多指令請看官方教學：https://git-scm.com/docs/gittutorial

## §13 課堂練習與期中銜接

### 🧪 課堂練習（任選，先完成者往下做）

| # | 練習 | 提示 |
|---|---|---|
| E1 | 把 `CATEGORIES` 換成你們期中登記的 3 個類別，重跑 §3 與 §7 | 至少 1 texture + 1 object |
| E2 | 在 `MODES` 新增 `"partial"`：只解凍 `layer4` + `fc` | 在 `run_experiment` 中依參數名稱 `name.startswith("layer4")` 決定 `requires_grad` |
| E3 | 把 `BACKBONE_MODE` 改成 `"finetune"`，排名改變了嗎？ | VGG16 在 Colab 上可把 `BATCH_SIZE` 調成 8 |
| E4 | 測試 augmentation 組合，例如 `"hflip+rot90"` 對 texture | 確認 §9.3 的觀察 |
| E5 | 把 `weight_decay` 改成 `0`、`0.01`、`0.1`（Topic 7–8） | 一次只改一件事 |
| E6 | 用 `IMG_SIZE = 256`、`CACHE_SIZE = 320` 重跑 | 小瑕疵在高解析度下比較好偵測嗎？ |
| E7 | 加入 `"crop"` 的較溫和版本 `scale=(0.8, 1.0)`，再跑一次 §9.2 | 標籤失真的比例下降多少？AUROC 呢？ |

### ✅ 期中 Track A Checklist（對應評分表）

**Experimental design（30%）**
- [ ] 3 個類別（≥1 texture、≥1 object），已在 Week 10 proposal 登記
- [ ] 官方 test set 50/50、依 defect type 分層；切分檔（CSV）放進 repo
- [ ] held-out 只在最後評估，**沒有**用來調參、早停、選門檻
- [ ] Baseline（無 augmentation）→ 每次只改一件事
- [ ] 每個數字都是 3 seeds 的 mean ± std（**不可只報最好的 seed**）

**Results & analysis（35%）**
- [ ] 比較 frozen vs fine-tune
- [ ] 至少 4 種 augmentation，並附 **≥1 張標籤失真的圖**
- [ ] AUROC、Recall@5%FPR、混淆矩陣 + 一句門檻理由
- [ ] 3 個 FN + 3 個 FP 的圖與解釋

**Reproducibility（15%）**
- [ ] README 寫清楚如何從頭重跑；seed、版本、超參數都有紀錄（本 notebook 的 `config.json`）

**Report & presentation（20%）**
- [ ] IEEE 格式 4 頁；第 5、6 節（augmentation 分析、錯誤分析）是得分重點

> 想挑戰 **Track B（Anomaly detection）**：只用官方 train set 的良品，搭配 anomalib 的 PatchCore / PaDiM。它同樣建立在「ImageNet 預訓練特徵」上（就是 §5.4 看到的那種特徵），只是不訓練分類器，而是量測測試圖的特徵離良品有多遠。請以 anomalib 官方文件的最新 API 為準。

### 參考文獻
- Bergmann et al. *MVTec AD — A Comprehensive Real-World Dataset for Unsupervised Anomaly Detection.* CVPR 2019；*The MVTec Anomaly Detection Dataset.* IJCV 2021.
- He et al. *Deep Residual Learning for Image Recognition.* CVPR 2016.
- Selvaraju et al. *Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization.* ICCV 2017.
- Yosinski et al. *How transferable are features in deep neural networks?* NeurIPS 2014.